In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🏗️ Infrastructure as Code (IaC) & Deployment Automation Guide

**Managing cloud infrastructure through code with Terraform, CloudFormation, and Bicep**

This guide covers:
- IaC fundamentals and benefits
- Terraform workflows and modules
- AWS CloudFormation
- Azure Bicep templates
- State management and remote backends
- GitOps and infrastructure pipelines
- Cost estimation and optimization
- Testing IaC
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Terraform Fundamentals

### HCL Syntax Basics
```hcl
# Provider configuration
terraform {
  required_version = ">= 1.5"
  required_providers {
    aws = {
      source  = "hashicorp/aws"
      version = "~> 5.0"
    }
  }
}

provider "aws" {
  region = var.aws_region
}

# Variables
variable "app_name" {
  type        = string
  description = "Application name"
  default     = "my-app"
}

# Resource definition
resource "aws_s3_bucket" "app_bucket" {
  bucket = "${var.app_name}-bucket-${data.aws_caller_identity.current.account_id}"
  
  tags = {
    Name        = var.app_name
    Environment = var.environment
    ManagedBy   = "terraform"
  }
}

# Output values
output "bucket_name" {
  value       = aws_s3_bucket.app_bucket.id
  description = "Name of S3 bucket"
}

# Data sources (read-only)
data "aws_caller_identity" "current" {}
```

### State Management
```hcl
# Remote backend (S3 + DynamoDB for locking)
terraform {
  backend "s3" {
    bucket         = "terraform-state"
    key            = "prod/terraform.tfstate"
    region         = "us-east-1"
    encrypt        = true
    dynamodb_table = "terraform-locks"
  }
}

# Local state (development only)
# terraform.tfstate and terraform.tfstate.backup created locally
```

### Terraform Workflow
```bash
# 1. Initialize working directory
terraform init

# 2. Validate configuration
terraform validate

# 3. Format code
terraform fmt -recursive

# 4. Plan changes (dry-run)
terraform plan -out=tfplan

# 5. Review plan
cat tfplan

# 6. Apply changes
terraform apply tfplan

# 7. Destroy resources (when done)
terraform destroy
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Modular Terraform Architecture

```hcl
# Root module structure
root/
├── main.tf           # Main resources
├── variables.tf      # Input variables
├── outputs.tf        # Output values
├── terraform.tfvars  # Variable values
├── modules/
│  ├── networking/
│  │  ├── main.tf
│  │  ├── variables.tf
│  │  └── outputs.tf
│  ├── compute/
│  └── database/
└── environments/
   ├── dev.tfvars
   ├── staging.tfvars
   └── prod.tfvars

# Main file using modules
module "networking" {
  source = "./modules/networking"
  
  app_name = var.app_name
  vpc_cidr = var.vpc_cidr
  
  tags = local.common_tags
}

module "compute" {
  source = "./modules/compute"
  
  app_name            = var.app_name
  instance_count      = var.instance_count
  instance_type       = var.instance_type
  subnet_ids          = module.networking.subnet_ids
  security_group_ids  = [module.networking.app_sg_id]
  
  depends_on = [module.networking]
}

# Module outputs
output "load_balancer_dns" {
  value = module.compute.load_balancer_dns
}
```

### Reusable VPC Module
```hcl
# modules/networking/main.tf
resource "aws_vpc" "main" {
  cidr_block           = var.vpc_cidr
  enable_dns_hostnames = true
  
  tags = merge(var.tags, { Name = "${var.app_name}-vpc" })
}

resource "aws_subnet" "public" {
  count             = length(var.availability_zones)
  vpc_id            = aws_vpc.main.id
  cidr_block        = cidrsubnet(var.vpc_cidr, 8, count.index)
  availability_zone = var.availability_zones[count.index]
  
  map_public_ip_on_launch = true
  
  tags = merge(var.tags, { Name = "${var.app_name}-public-${count.index}" })
}

resource "aws_internet_gateway" "main" {
  vpc_id = aws_vpc.main.id
  
  tags = merge(var.tags, { Name = "${var.app_name}-igw" })
}

# modules/networking/variables.tf
variable "app_name" {
  type = string
}

variable "vpc_cidr" {
  type    = string
  default = "10.0.0.0/16"
}

variable "availability_zones" {
  type    = list(string)
  default = ["us-east-1a", "us-east-1b", "us-east-1c"]
}

variable "tags" {
  type    = map(string)
  default = {}
}

# modules/networking/outputs.tf
output "vpc_id" {
  value = aws_vpc.main.id
}

output "subnet_ids" {
  value = aws_subnet.public[*].id
}
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. AWS CloudFormation & Azure Bicep

### CloudFormation YAML
```yaml
AWSTemplateFormatVersion: '2010-09-09'
Description: 'ECS Fargate Application Stack'

Parameters:
  AppName:
    Type: String
    Default: my-app
  Environment:
    Type: String
    AllowedValues: [dev, staging, prod]

Conditions:
  IsProd: !Equals [!Ref Environment, prod]

Resources:
  # VPC
  AppVPC:
    Type: AWS::EC2::VPC
    Properties:
      CidrBlock: 10.0.0.0/16
      EnableDnsHostnames: true
      Tags:
        - Key: Name
          Value: !Sub '${AppName}-vpc'

  # ECS Cluster
  ECSCluster:
    Type: AWS::ECS::Cluster
    Properties:
      ClusterName: !Sub '${AppName}-cluster'

  # Task Definition
  TaskDefinition:
    Type: AWS::ECS::TaskDefinition
    Properties:
      Family: !Sub '${AppName}-task'
      NetworkMode: awsvpc
      RequiresCompatibilities: [FARGATE]
      Cpu: !If [IsProd, '512', '256']
      Memory: !If [IsProd, '1024', '512']
      ContainerDefinitions:
        - Name: app
          Image: !Sub '${AWS::AccountId}.dkr.ecr.${AWS::Region}.amazonaws.com/${AppName}:latest'
          PortMappings:
            - ContainerPort: 8000
          LogConfiguration:
            LogDriver: awslogs
            Options:
              awslogs-group: !Ref LogGroup

  LogGroup:
    Type: AWS::Logs::LogGroup
    Properties:
      LogGroupName: !Sub '/ecs/${AppName}'
      RetentionInDays: !If [IsProd, 30, 7]

Outputs:
  ClusterName:
    Value: !Ref ECSCluster
    Export:
      Name: !Sub '${AppName}-cluster'
```

### Azure Bicep
```bicep
param location string = resourceGroup().location
param appName string = 'my-app'
param environment string = 'dev'

// Variables
var vmSize = environment == 'prod' ? 'Standard_D4s_v3' : 'Standard_B2s'
var commonTags = {
  application: appName
  environment: environment
  managedBy: 'bicep'
}

// Virtual Network
resource virtualNetwork 'Microsoft.Network/virtualNetworks@2023-04-01' = {
  name: '${appName}-vnet'
  location: location
  properties: {
    addressSpace: {
      addressPrefixes: [
        '10.0.0.0/16'
      ]
    }
    subnets: [
      {
        name: 'default'
        properties: {
          addressPrefix: '10.0.1.0/24'
        }
      }
    ]
  }
  tags: commonTags
}

// Virtual Machine
resource virtualMachine 'Microsoft.Compute/virtualMachines@2023-03-01' = {
  name: '${appName}-vm'
  location: location
  properties: {
    hardwareProfile: {
      vmSize: vmSize
    }
    osProfile: {
      computerName: appName
      adminUsername: 'azureuser'
    }
    storageProfile: {
      imageReference: {
        publisher: 'Canonical'
        offer: '0001-com-ubuntu-server-focal'
        sku: '20_04-lts-gen2'
        version: 'latest'
      }
    }
  }
  tags: commonTags
}

// Outputs
output vmId string = virtualMachine.id
output vnetId string = virtualNetwork.id
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Testing & Validation IaC

```python
# test_infrastructure.py - Test IaC configurations
import json
import subprocess
import pytest

class TestTerraformConfiguration:
    
    def setup_method(self):
        """Initialize Terraform"""
        subprocess.run(['terraform', 'init'], check=True)
    
    def test_terraform_validate(self):
        """Validate Terraform syntax"""
        result = subprocess.run(
            ['terraform', 'validate'],
            capture_output=True,
            text=True
        )
        assert result.returncode == 0, f"Validation failed: {result.stderr}"
    
    def test_terraform_plan_succeeds(self):
        """Test that plan generation succeeds"""
        result = subprocess.run(
            ['terraform', 'plan', '-json'],
            capture_output=True,
            text=True
        )
        assert result.returncode == 0
        
        # Parse JSON output
        lines = result.stdout.strip().split('\n')
        for line in lines:
            event = json.loads(line)
            # Check for errors
            assert event.get('type') != 'diagnostic' or event.get('severity') != 'error'
    
    def test_resource_count(self):
        """Verify expected number of resources"""
        result = subprocess.run(
            ['terraform', 'plan', '-json'],
            capture_output=True,
            text=True
        )
        
        resource_changes = 0
        for line in result.stdout.strip().split('\n'):
            event = json.loads(line)
            if event.get('type') == 'resource_drift':
                resource_changes += 1
        
        # Adjust threshold as needed
        assert resource_changes < 50, "Too many resource changes"
    
    def test_security_groups_restricted(self):
        """Verify security groups don't allow unrestricted access"""
        result = subprocess.run(
            ['terraform', 'show', '-json'],
            capture_output=True,
            text=True
        )
        
        state = json.loads(result.stdout)
        
        for resource in state.get('values', {}).get('root_module', {}).get('resources', []):
            if 'security_group' in resource['type']:
                rules = resource['values'].get('ingress', [])
                
                for rule in rules:
                    # Check for unrestricted CIDR
                    if rule.get('cidr_blocks') == ['0.0.0.0/0']:
                        assert rule.get('from_port') != 22, "SSH open to world"

class TestCloudFormationTemplate:
    
    def test_cfn_validate(self):
        """Validate CloudFormation template"""
        result = subprocess.run([
            'aws', 'cloudformation', 'validate-template',
            '--template-body', 'file://template.yaml'
        ], capture_output=True)
        
        assert result.returncode == 0
    
    def test_cfn_parameters(self):
        """Verify all required parameters have defaults or descriptions"""
        import yaml
        
        with open('template.yaml') as f:
            template = yaml.safe_load(f)
        
        for param_name, param_def in template.get('Parameters', {}).items():
            # Every parameter should have a description
            assert 'Description' in param_def, f"Parameter {param_name} missing description"
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 5. GitOps & Infrastructure Pipelines

### GitHub Actions for IaC
```yaml
# .github/workflows/terraform.yml
name: Terraform CI/CD

on:
  push:
    branches: [main]
    paths: ['terraform/**']
  pull_request:
    branches: [main]
    paths: ['terraform/**']

jobs:
  terraform:
    runs-on: ubuntu-latest
    
    steps:
      - uses: actions/checkout@v3
      
      - uses: hashicorp/setup-terraform@v2
        with:
          terraform_version: 1.5.0
      
      - name: Terraform Format
        run: terraform fmt -check -recursive
      
      - name: Terraform Init
        run: terraform init -upgrade
        env:
          AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
          AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
      
      - name: Terraform Validate
        run: terraform validate
      
      - name: Terraform Plan
        run: terraform plan -out=tfplan
        env:
          AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
          AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
      
      - name: Upload Plan
        uses: actions/upload-artifact@v3
        with:
          name: tfplan
          path: tfplan
  
  apply:
    runs-on: ubuntu-latest
    needs: terraform
    if: github.ref == 'refs/heads/main' && github.event_name == 'push'
    
    steps:
      - uses: actions/checkout@v3
      - uses: hashicorp/setup-terraform@v2
      
      - name: Download Plan
        uses: actions/download-artifact@v3
        with:
          name: tfplan
      
      - name: Terraform Apply
        run: terraform apply -auto-approve tfplan
        env:
          AWS_ACCESS_KEY_ID: ${{ secrets.AWS_ACCESS_KEY_ID }}
          AWS_SECRET_ACCESS_KEY: ${{ secrets.AWS_SECRET_ACCESS_KEY }}
```

### Best Practices Checklist
```
✅ IaC Organization
- [ ] Modular structure with reusable modules
- [ ] Separate environments (dev, staging, prod)
- [ ] Variable defaults for safety
- [ ] Comprehensive outputs

✅ State Management
- [ ] Remote backend configured
- [ ] State locking enabled
- [ ] Encryption at rest enabled
- [ ] State backup strategy

✅ Security
- [ ] No hardcoded secrets
- [ ] Least privilege IAM roles
- [ ] Encryption enabled
- [ ] Restricted security groups

✅ Testing & Validation
- [ ] Terraform validate runs
- [ ] Format checks pass
- [ ] Cost estimation reviewed
- [ ] Pre-deployment tests pass

✅ CI/CD Integration
- [ ] Plan runs on PRs
- [ ] Approval required before apply
- [ ] Audit logs enabled
- [ ] Rollback procedure documented
```
</VSCode.Cell>
```